# Transfer-Learning Intervals: Global vs Grouped Pooling

Temporary notebook to compare pooled conformal intervals for new transfer-learning items:
- global pooling over all calibrated ids
- grouped pooling with categorical static features


In [1]:
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals, generate_daily_series

try:
    from datasetsforecast.m5 import M5
    HAS_M5 = True
except Exception:
    HAS_M5 = False

HAS_M5


True

In [2]:
h = 14

if HAS_M5:
    y_df, _, s_df = M5.load(directory="data")
    y_df["ds"] = pd.to_datetime(y_df["ds"])
    df = y_df.merge(s_df, on="unique_id", how="left")
    all_ids = df["unique_id"].drop_duplicates().tolist()

    # Pure transfer setting: no overlap between old and new ids.
    n_base = min(200, max(1, len(all_ids) - 50))
    n_transfer = min(50, len(all_ids) - n_base)
    base_ids = set(all_ids[:n_base])
    transfer_ids = set(all_ids[n_base : n_base + n_transfer])

    static_features = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
    interval_groupby = ["state_id", "cat_id"]
    base_df = df[df["unique_id"].isin(base_ids)].copy()
    transfer_all = df[df["unique_id"].isin(transfer_ids)].copy()
else:
    # Synthetic fallback with disjoint old/new ids and two categorical static features.
    def add_groups(tmp, prefix):
        out = tmp.copy()
        out["unique_id"] = prefix + out["unique_id"].astype(str)
        idx = out["unique_id"].str.extract(r"(\d+)").astype(int)[0]
        out["cat0"] = np.where(idx % 2 == 0, "A", "B")
        out["cat1"] = np.where((idx // 2) % 2 == 0, "X", "Y")
        out["y"] = out["y"] * np.where(out["cat0"] == "B", 8.0, 1.0)
        out["cat0"] = out["cat0"].astype("category")
        out["cat1"] = out["cat1"].astype("category")
        return out

    base_df = add_groups(generate_daily_series(4, 120, 120, equal_ends=True, seed=0), prefix="old_")
    transfer_all = add_groups(generate_daily_series(6, 120, 120, equal_ends=True, seed=1), prefix="new_")
    static_features = ["cat0", "cat1"]
    interval_groupby = ["cat0", "cat1"]

transfer_valid = transfer_all.groupby("unique_id", observed=True).tail(h)
transfer_train = transfer_all.drop(transfer_valid.index)

# Old-id holdout for normal (non-transfer) conformal baseline.
base_old_valid = base_df.groupby("unique_id", observed=True).tail(h)
base_old_train = base_df.drop(base_old_valid.index)

old_ids = sorted(base_df["unique_id"].drop_duplicates())
new_ids = sorted(transfer_train["unique_id"].drop_duplicates())

assert set(old_ids).isdisjoint(set(new_ids)), "old_ids and new_ids should be disjoint"

len(old_ids), len(new_ids), len(base_df), len(transfer_train)


(200, 50, 357380, 80581)

In [3]:
fit_kwargs_transfer = dict(
    static_features=static_features,
    prediction_intervals=PredictionIntervals(n_windows=2, h=h, method="conformal_distribution"),
)

# 1) Transfer-learning + global pooled fallback (new ids only)
fcst_global = MLForecast(
    models=lgb.LGBMRegressor(random_state=0, verbosity=-1),
    freq="D",
    lags=[1, 7, 28],
    num_threads=1,
).fit(base_df, **fit_kwargs_transfer)

with warnings.catch_warnings(record=True) as caught_global:
    warnings.simplefilter("always")
    preds_global = fcst_global.predict(h=h, new_df=transfer_train, level=[80])

# 2) Transfer-learning + grouped pooled fallback (new ids only)
fcst_grouped = MLForecast(
    models=lgb.LGBMRegressor(random_state=0, verbosity=-1),
    freq="D",
    lags=[1, 7, 28],
    num_threads=1,
).fit(base_df, **fit_kwargs_transfer)

with warnings.catch_warnings(record=True) as caught_grouped:
    warnings.simplefilter("always")
    preds_grouped = fcst_grouped.predict(
        h=h,
        new_df=transfer_train,
        level=[80],
        interval_groupby=interval_groupby,
    )

# 3) Normal conformal prediction on old ids (distribution / error / global)
pi = fit_kwargs_transfer["prediction_intervals"]

def fit_old_with_method(method):
    fit_kwargs = dict(
        static_features=static_features,
        prediction_intervals=PredictionIntervals(n_windows=pi.n_windows, h=pi.h, method=method),
    )
    return MLForecast(
        models=lgb.LGBMRegressor(random_state=0, verbosity=-1),
        freq="D",
        lags=[1, 7, 28],
        num_threads=1,
    ).fit(base_old_train, **fit_kwargs)

fcst_old_dist = fit_old_with_method("conformal_distribution")
fcst_old_error = fit_old_with_method("conformal_error")
fcst_old_global = fit_old_with_method("global")

preds_old_dist = fcst_old_dist.predict(h=h, level=[80], ids=old_ids)
preds_old_error = fcst_old_error.predict(h=h, level=[80], ids=old_ids)
preds_old_global = fcst_old_global.predict(h=h, level=[80], ids=old_ids)

print("global warnings:", [str(w.message) for w in caught_global])
print("grouped warnings:", [str(w.message) for w in caught_grouped])


global warnings: ['Prediction intervals were calibrated on a different set of series than the current transfer-learning state. Falling back to pooled conformity scores.']
grouped warnings: ['Prediction intervals were calibrated on a different set of series than the current transfer-learning state. Falling back to pooled conformity scores.']


In [4]:
def get_interval_cols(df, level=80):
    lo_suffix = f"-lo-{level}"
    hi_suffix = f"-hi-{level}"
    lo_candidates = [c for c in df.columns if str(c).endswith(lo_suffix)]
    hi_candidates = [c for c in df.columns if str(c).endswith(hi_suffix)]
    if not lo_candidates or not hi_candidates:
        raise ValueError(
            "Could not find interval columns for level 80. "
            f"Available columns: {list(df.columns)}"
        )
    lo_col = lo_candidates[0]
    model_name = lo_col[: -len(lo_suffix)]
    hi_col = f"{model_name}{hi_suffix}"
    if hi_col not in df.columns:
        raise ValueError(
            f"Found lo column {lo_col} but missing matching hi column {hi_col}. "
            f"Available columns: {list(df.columns)}"
        )
    return lo_col, hi_col


def summarize(eval_df, setting, cohort_name, ids):
    lo_col, hi_col = get_interval_cols(eval_df, level=80)
    subset = eval_df[eval_df["unique_id"].isin(ids)].copy()
    coverage = ((subset["y"] >= subset[lo_col]) & (subset["y"] <= subset[hi_col])).mean()
    avg_width = (subset[hi_col] - subset[lo_col]).mean()
    return pd.DataFrame(
        [
            {
                "setting": setting,
                "cohort": cohort_name,
                "n_ids": len(ids),
                "n_rows": len(subset),
                "coverage_80": coverage,
                "avg_interval_width_80": avg_width,
                "interval_cols": f"{lo_col} | {hi_col}",
            }
        ]
    )


eval_global = preds_global.merge(
    transfer_valid[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="left",
)
eval_grouped = preds_grouped.merge(
    transfer_valid[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="left",
)

def eval_old(preds):
    return preds.merge(
        base_old_valid[["unique_id", "ds", "y"]],
        on=["unique_id", "ds"],
        how="left",
    )

eval_old_dist = eval_old(preds_old_dist)
eval_old_error = eval_old(preds_old_error)
eval_old_global = eval_old(preds_old_global)

results = pd.concat(
    [
        summarize(eval_global, "transfer_global_pooling", "new_items", new_ids),
        summarize(eval_grouped, "transfer_grouped_pooling", "new_items", new_ids),
        summarize(eval_old_dist, "old_conformal_distribution", "old_items", old_ids),
        summarize(eval_old_error, "old_conformal_error", "old_items", old_ids),
        summarize(eval_old_global, "old_global_pooling", "old_items", old_ids),
    ],
    ignore_index=True,
)

results.sort_values(["cohort", "setting"]).reset_index(drop=True)


,setting,cohort,n_ids,n_rows,coverage_80,avg_interval_width_80,interval_cols
0,transfer_global_pooling,new_items,50,700,0.861429,4.238164,LGBMRegressor-lo-80 | LGBMRegressor-hi-80
1,transfer_grouped_pooling,new_items,50,700,0.860000,4.104752,LGBMRegressor-lo-80 | LGBMRegressor-hi-80
2,old_conformal_distribution,old_items,200,2800,0.585714,3.846582,LGBMRegressor-lo-80 | LGBMRegressor-hi-80
3,old_conformal_error,old_items,200,2800,0.607500,4.110334,LGBMRegressor-lo-80 | LGBMRegressor-hi-80
4,old_global_pooling,old_items,200,2800,0.797857,4.158869,LGBMRegressor-lo-80 | LGBMRegressor-hi-80
